# Basic Information Retrieval System Using CISI Dataset

## Step 1: Import Libraries
This step imports the required libraries for text processing and similarity calculation.


In [39]:
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Step 2: Load CISI Documents
# This reads the CISI.ALL dataset and creates a DataFrame of document records.
def load_cisi_documents(file_path):
    documents = []
    with open(file_path, "r", encoding="latin1") as file:
        content = file.read()
    records = content.split(".I ")
    for record in records[1:]:
        lines = record.splitlines()
        doc_id = lines[0].strip()
        text = " ".join(lines[1:])
        text = re.sub(r"\.[A-Z]\s*", " ", text)
        documents.append({
            "doc_id": doc_id,
            "text": text
        })
    return pd.DataFrame(documents)
documents_df = load_cisi_documents("Dataset/CISI.ALL")
print(documents_df.head())
print("Number of documents:", len(documents_df))

  doc_id                                               text
0      1   18 Editions of the Dewey Decimal Classificati...
1      2   Use Made of Technical Libraries  Slater, M.  ...
2      3   Two Kinds of Power An Essay on Bibliographic ...
3      4   Systems Analysis of a University Library;  fi...
4      5   A Library Management Game: a report on a rese...
Number of documents: 1460


In [47]:
# Step 3: Preprocess Documents
# This lowercases text and removes punctuation and repeated spaces.

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


documents_df["clean_text"] = documents_df["text"].apply(preprocess_text)
print(documents_df[["doc_id", "clean_text"]].head())

  doc_id                                         clean_text
0      1  editions of the dewey decimal classifications ...
1      2  use made of technical libraries slater m this ...
2      3  two kinds of power an essay on bibliographic c...
3      4  systems analysis of a university library final...
4      5  a library management game a report on a resear...


In [48]:
# Step 4: Create TF-IDF Index
# This builds the vector space model for the documents.

vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(documents_df["clean_text"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (1460, 5665)


In [ ]:
# Step 5: Process User Query
# This function preprocesses a query and compares it with all document vectors.

def search(query, top_k=10):
    query = preprocess_text(query)
    query_vector = vectorizer.transform([query])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = similarity_scores.argsort()[::-1][:top_k]

    results = documents_df.iloc[top_indices][["doc_id", "text"]].copy()
    results["similarity_score"] = similarity_scores[top_indices]

    return results

In [49]:
# Step 6: Test the Retrieval System
# This runs a sample query and prints the most relevant documents.

query = "information retrieval system"
results = search(query, top_k=10)
print(results[["doc_id", "similarity_score"]])

     doc_id  similarity_score
538     539          0.449957
1135   1136          0.422254
564     565          0.422171
453     454          0.379874
460     461          0.373163
458     459          0.371998
894     895          0.346415
174     175          0.342326
850     851          0.331990
508     509          0.330252
